In [1]:
# diferent conda environment -> This notebook for QGISPy
import sys
import os

import qgis
from qgis.gui import *
from qgis.core import *
from qgis.utils import plugins
from PyQt5.QtCore import *
from qgis.analysis import QgsNativeAlgorithms

sys.path.append('/usr/share/qgis/python/plugins/')
sys.path.append('/usr/share/qgis/python/')

QgsApplication.setPrefixPath('/usr', True)
app = QgsApplication([], False)
app.initQgis()




In [2]:
import processing
from processing.core.Processing import Processing
Processing.initialize()

QgsApplication.processingRegistry().addProvider(QgsNativeAlgorithms())
for alg in QgsApplication.processingRegistry().algorithms():
        print(alg.id(), "--->", alg.displayName())

3d:tessellate ---> Tessellate
gdal:aspect ---> Aspect
gdal:assignprojection ---> Assign projection
gdal:buffervectors ---> Buffer vectors
gdal:buildvirtualraster ---> Build virtual raster
gdal:buildvirtualvector ---> Build virtual vector
gdal:cliprasterbyextent ---> Clip raster by extent
gdal:cliprasterbymasklayer ---> Clip raster by mask layer
gdal:clipvectorbyextent ---> Clip vector by extent
gdal:clipvectorbypolygon ---> Clip vector by mask layer
gdal:colorrelief ---> Color relief
gdal:contour ---> Contour
gdal:contour_polygon ---> Contour Polygons
gdal:convertformat ---> Convert format
gdal:dissolve ---> Dissolve
gdal:executesql ---> Execute SQL
gdal:extractprojection ---> Extract projection
gdal:fillnodata ---> Fill nodata
gdal:gdal2tiles ---> gdal2tiles
gdal:gdal2xyz ---> gdal2xyz
gdal:gdalinfo ---> Raster information
gdal:gridaverage ---> Grid (Moving average)
gdal:griddatametrics ---> Grid (Data metrics)
gdal:gridinversedistance ---> Grid (Inverse distance to a power)
gdal:grid

Logged warning: Duplicate provider native registered


In [52]:
input_raster_slope = QgsRasterLayer('/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/slope_clean_class.tif')#,
                              #'class_slope_demnas') 

#input_raster_slope = QgsRasterLayer('/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/class_slope_demnas.map')
    
input_raster_dem = QgsRasterLayer('/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/dem_utm_for_slope.tif')#,
                              #'dem_utm_for_slope') 

    
entries = []

In [53]:
from qgis.analysis import QgsRasterCalculator, QgsRasterCalculatorEntry

In [54]:
class_slope_demnas = QgsRasterCalculatorEntry()
class_slope_demnas.ref = 'class_slope_demnas@1'
class_slope_demnas.raster = input_raster_slope
class_slope_demnas.badNumber = 1
entries.append(class_slope_demnas)


In [55]:
dem_utm_for_slope = QgsRasterCalculatorEntry()
dem_utm_for_slope.ref = 'dem_utm_for_slope@1'
dem_utm_for_slope.raster = input_raster_dem
dem_utm_for_slope.badNumber = 1
entries.append(dem_utm_for_slope)

In [56]:
def dem_per_slope_class(output,i):
    calc = QgsRasterCalculator(f'if("class_slope_demnas@1"={i},"dem_utm_for_slope@1",0/0)',
                            #'("class_slope_demnas@1" = 1) * "dem_utm_for_slope@1"',

                          output, 'GTiff', input_raster_dem.extent(), input_raster_dem.width(),
                           input_raster_dem.height(),
                           entries
                          )

    calc.processCalculation()
    print(f'calculated for the output: {output}')
    return output



In [57]:
import os

In [58]:
base_location = '/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis'
list_class = [1,2,3,4,5]

list_output = [os.path.join(base_location, f'dem_class_{str(i)}') for i in list_class]

for i in range(len(list_output)):
    dem_per_slope_class(list_output[i],list_class[i])

calculated for the output: /home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/dem_class_1
calculated for the output: /home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/dem_class_2
calculated for the output: /home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/dem_class_3
calculated for the output: /home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/dem_class_4
calculated for the output: /home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/dem_class_5


/tmp/ipykernel_221421/3872326197.py:2: DeprecationWarning: QgsRasterCalculator constructor is deprecated
  calc = QgsRasterCalculator(f'if("class_slope_demnas@1"={i},"dem_utm_for_slope@1",0/0)',


In [59]:
# does not work still
# processing.run("qgis:rastercalculator", 
               
#                {#'EXPRESSION':'if("class_slope_demnas@1"=1,"dem_utm_for_slope@1",0/0)',
#                 'EXPRESSION':'("class_slope_demnas@1" = 1) * "dem_utm_for_slope@1"',
#                 'LAYERS':['/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/slope_clean_class.tif',],
#                          #'/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/dem_utm_for_slope.tif'],
#                 #'LAYERS': entries,
#                 #'CELLSIZE':0,
#                 #'EXTENT':None,
#                 #'CRS':'epsg:4326',
#                 'OUTPUT':'/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/test_dem.tif'})

In [60]:
list_interval = [0.5,1,1.5,2,2.5]
list_output_contour = [os.path.join(base_location, f'contour_interval_{str(i).replace(".","_")}.gpkg') for i in list_interval]

for i in range(len(list_output)):
    processing.run("gdal:contour", 
                   {'INPUT':list_output[i],
                    'BAND':1,
                    'INTERVAL':list_interval[i],
                    'FIELD_NAME':'ELEV',
                    'CREATE_3D':False,
                    'IGNORE_NODATA':False,
                    'NODATA':None,
                    'OFFSET':0,
                    'EXTRA':'',
                    'OUTPUT':list_output_contour[i]})

In [61]:
list_output_contour

['/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/contour_interval_0_5.gpkg',
 '/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/contour_interval_1.gpkg',
 '/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/contour_interval_1_5.gpkg',
 '/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/contour_interval_2.gpkg',
 '/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/contour_interval_2_5.gpkg']

In [62]:
output_result_contour = os.path.join(base_location,'contour_all_class.gpkg')

# merge-all_into_one
processing.run("native:mergevectorlayers", {'LAYERS': list_output_contour,
                                            'CRS':None,'OUTPUT':output_result_contour})

{'OUTPUT': '/home/mfirdaus/TREEO/TREEO/IQBAL_DATA/VM_DATA_SHARED/GIS_ArcGISPro/TREEO/T4T/dem_analysis/contour_all_class.gpkg'}

In [ ]:
# Stop QGIS appllication
app.exitQgis()
app.exit()
